# 03 · Mapeo de campos usados en el cálculo

## Objetivo
Listar, hoja por hoja, el valor cacheado de los campos que entran al cálculo, con el rótulo de BS&W separado del número.

## Alcance
No se imputa ningún campo y no se convierte de unidad. El esquema común de otros Excel no existe todavía; este mapeo vale solo para BES-jun.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

import openpyxl
from ingesta.leer_bes_jun import leer_libro
hojas = leer_libro(libro)
columnas = [
    "pozo", "api_deg", "bsw_numero", "bsw_unidad_rotulo", "densidad_agua_lb_gal",
    "temperatura_c", "presion_sensor_psi", "thp_psi", "caudal_operacion_bfpd",
    "caudal_prueba_bfpd", "id_tubing_in", "tvd_intake_ft", "tvd_sensor_ft",
]
for h in hojas:
    print("---", h["hoja"], h["pozo"])
    for col in columnas:
        print(f"  {col}={h[col]}")
formulas = openpyxl.load_workbook(libro, data_only=False)
hoja = formulas["QF-378HST"]
print("formula_Q_prueba", "QF-378HST!G23", hoja["G23"].value)
print("formula_Q_operacion", "QF-378HST!B68", hoja["B68"].value)
formulas.close()


--- QF-554H QF-084
  pozo=QF-084
  api_deg=12.0
  bsw_numero=0.9755
  bsw_unidad_rotulo=%
  densidad_agua_lb_gal=8.345
  temperatura_c=65.55555555555556
  presion_sensor_psi=732.0
  thp_psi=183.0
  caudal_operacion_bfpd=10301.0
  caudal_prueba_bfpd=10301.0
  id_tubing_in=2.996
  tvd_intake_ft=2653.176659090909
  tvd_sensor_ft=2678.821959090909
--- QF-545H QF-301
  pozo=QF-301
  api_deg=12.0
  bsw_numero=0.9766
  bsw_unidad_rotulo=%
  densidad_agua_lb_gal=8.345
  temperatura_c=62.22222222222222
  presion_sensor_psi=667.0
  thp_psi=210.0
  caudal_operacion_bfpd=8767.0
  caudal_prueba_bfpd=8767.0
  id_tubing_in=2.996
  tvd_intake_ft=2534.603750819672
  tvd_sensor_ft=2553.917357377049
--- QF-500H QF-147
  pozo=QF-147
  api_deg=12.0
  bsw_numero=0.9644
  bsw_unidad_rotulo=%
  densidad_agua_lb_gal=8.345
  temperatura_c=65.55555555555556
  presion_sensor_psi=821.0
  thp_psi=194.0
  caudal_operacion_bfpd=7728.0
  caudal_prueba_bfpd=7728.0
  id_tubing_in=2.996
  tvd_intake_ft=2578.9682203125003

formula_Q_prueba QF-378HST!G23 =295.6*24
formula_Q_operacion QF-378HST!B68 =296.5*24


## Campos y unidades que el lector conserva
`api_deg` está en °API. `presion_sensor_psi` y `thp_psi` están en psi. Los caudales están en BFPD. `id_tubing_in` está en pulgadas. Las TVD están en ft. `bsw_unidad_rotulo` se imprime aparte del número: el rótulo es `%` y el número, entre 0.9557 y 0.9766, es el que la fórmula de densidad multiplica. No se divide entre 100.

En cuatro hojas el caudal de prueba cacheado es igual al de operación. En QF-378HST, pozo QF-005, no lo es. La celda `G23` contiene `=295.6*24` y la celda `B68` contiene `=296.5*24`. El factor 24 no está definido en otra celda leída por este cuaderno. Los cache son 7094.4 BFPD y 7116 BFPD. El cálculo posterior usa el caudal de operación, no el de prueba.

API es 12 en las cinco hojas. La presión de sensor va de 607 psi a 821 psi. Esos valores se listan en la salida; no se promedian entre hojas.
